In [4]:
"""
DeePiCt post-processing parameter search
=========================================
Goals:
  • Re-run clustering directly on stored probability_map.mrc for each fold
    (no retraining needed).
  • Grid-search threshold / min_cluster_size / max_cluster_size for hsp60
    depth-2 (leave-one-out) and ribosome depth-2 (2-fold).
  • Compare best tuned F1 vs. DeepETPicker baseline.
"""
import sys, pickle
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import mrcfile
from scipy.ndimage import label as scipy_label
from skimage.measure import regionprops_table

sys.path.append("./srcs")
import utils

# ── Ground-truth centers ────────────────────────────────────────────────────
with open("./release_assets/metadata/new_hsp60/centers3.pkl", "rb") as f:
    hsp60_centers = pickle.load(f)

with open("/virtual_data/train_data/training/cr/temp/ribo_centers_new.pkl", "rb") as f:
    ribo_centers = pickle.load(f)

MATCH_THRESHOLD = 20

# ── Fast post-processing helpers (NO retraining) ────────────────────────────
def compute_components(prob_vol: np.ndarray, threshold: float, connectivity: int = 3):
    """
    Compute connected components once and return per-component centroids + sizes.
    """
    binary = (prob_vol >= threshold).astype(np.uint8)
    struct = np.ones((3, 3, 3), dtype=np.uint8) if connectivity == 3 else None
    labeled, num = scipy_label(binary, structure=struct)
    if num == 0:
        return np.empty((0, 3), dtype=np.float32), np.empty((0,), dtype=np.int32)

    props = regionprops_table(labeled, properties=("area", "centroid"))
    centroids = np.column_stack([
        props["centroid-0"],
        props["centroid-1"],
        props["centroid-2"],
    ]).astype(np.float32)
    areas = np.asarray(props["area"], dtype=np.int32)
    return centroids, areas


def build_threshold_cache(prob_file: Path, thresholds, connectivity: int = 3):
    """
    Load one probability map once; precompute components for each threshold.
    """
    with mrcfile.open(str(prob_file), permissive=True) as m:
        vol = m.data.astype(np.float32)

    cache = {}
    for thr in thresholds:
        cache[thr] = compute_components(vol, thr, connectivity=connectivity)
    return cache


def score_from_components(centroids: np.ndarray, areas: np.ndarray, gt_centers: np.ndarray,
                          min_cs: int, max_cs: int, match_thr: int = 20) -> dict:
    keep = (areas > min_cs) & (areas <= max_cs)
    picked = centroids[keep]

    if len(picked) == 0:
        pred_n = 0
        tp = 0
    else:
        _, dists, _ = utils.match_and_find_closest(picked, gt_centers)
        tp = int(np.sum(dists < match_thr))
        pred_n = len(picked)

    gt_n = len(gt_centers)
    fp = pred_n - tp
    fn = gt_n - tp
    prec = tp / pred_n if pred_n else 0.0
    rec = tp / gt_n if gt_n else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(tp=tp, fp=fp, fn=fn, pred=pred_n, gt=gt_n, precision=prec, recall=rec, f1=f1)


print("Setup complete.")
print("Mode: post-processing only (no model retraining).")
print(f"  hsp60 tomos : {sorted(hsp60_centers.keys())}")
print(f"  ribo  tomos : {sorted(ribo_centers.keys())}")

Setup complete.
Mode: post-processing only (no model retraining).
  hsp60 tomos : ['24092001', '24092002', '24092018', '24092024', '24092028', '24092031']
  ribo  tomos : ['24092002', '24092009', '24092013', '24092014', '24092016', '24092017', '24092020', '24092023', '24092024', '24092025', '24092026', '24092029', '24092030', '24092031']


In [5]:
# ── hsp60 depth-2 LOO: probability-map paths ────────────────────────────────
hsp60_base = Path("/virtual_data/train_data/training/deepict/leave_one_out_hsp60_depth2")

hsp60_prob_maps = {}
for fold_dir in sorted(hsp60_base.glob("hsp60d2_loo_*_out")):
    for mrc in fold_dir.glob("predictions/*/*/hsp60/probability_map.mrc"):
        tomo_id = mrc.parents[1].name
        hsp60_prob_maps[tomo_id] = mrc

print("hsp60 probability maps found:")
for k, v in sorted(hsp60_prob_maps.items()):
    print(f"  {k}: {v}")

missing = sorted(set(hsp60_centers.keys()) - set(hsp60_prob_maps.keys()))
if missing:
    print(f"\nWARNING - no probability map for: {missing}")

# ── Grid-search parameters ───────────────────────────────────────────────────
thresholds      = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
min_cluster_sz  = [1, 3, 5, 10, 20]
max_cluster_sz  = [100, 200, 400, 800, 2000]

print("\nBuilding per-threshold component cache (post-process only)...")
hsp60_cache = {}
for tomo_id, prob_file in sorted(hsp60_prob_maps.items()):
    if tomo_id not in hsp60_centers:
        continue
    hsp60_cache[tomo_id] = build_threshold_cache(prob_file, thresholds, connectivity=3)

grid_records = []

total = len(thresholds) * len(min_cluster_sz) * len(max_cluster_sz)
print(f"\nRunning {total} parameter combinations ...")

for thr, mn, mx in product(thresholds, min_cluster_sz, max_cluster_sz):
    if mn >= mx:
        continue
    micro_tp = micro_fp = micro_fn = 0
    for tomo_id in sorted(hsp60_cache.keys()):
        gt = np.array(hsp60_centers[tomo_id], dtype=np.float32)
        centroids, areas = hsp60_cache[tomo_id][thr]
        s = score_from_components(centroids, areas, gt, mn, mx, MATCH_THRESHOLD)
        micro_tp += s["tp"]
        micro_fp += s["fp"]
        micro_fn += s["fn"]

    p = micro_tp / (micro_tp + micro_fp) if (micro_tp + micro_fp) else 0.0
    r = micro_tp / (micro_tp + micro_fn) if (micro_tp + micro_fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    grid_records.append(dict(threshold=thr, min_cluster=mn, max_cluster=mx,
                              micro_tp=micro_tp, micro_fp=micro_fp, micro_fn=micro_fn,
                              precision=p, recall=r, f1=f1))

hsp60_grid = pd.DataFrame(grid_records).sort_values("f1", ascending=False).reset_index(drop=True)

print("\nTop-20 hsp60 parameter combinations by micro-F1:")
print(hsp60_grid.head(20)[["threshold","min_cluster","max_cluster","precision","recall","f1"]].to_string(index=False))

best_hsp60 = hsp60_grid.iloc[0]
print(f"\nBest hsp60 -> threshold={best_hsp60.threshold} min_cluster={int(best_hsp60.min_cluster)} "
      f"max_cluster={int(best_hsp60.max_cluster)}")
print(f"  micro-F1={best_hsp60.f1:.4f} precision={best_hsp60.precision:.4f} recall={best_hsp60.recall:.4f}")
print("  (baseline 0.5/5/400 = 0.1165)")

# ── Per-tomo breakdown with best params ─────────────────────────────────────
print("\nPer-tomo breakdown at best params:")
best_rows = []
for tomo_id in sorted(hsp60_cache.keys()):
    gt = np.array(hsp60_centers[tomo_id], dtype=np.float32)
    centroids, areas = hsp60_cache[tomo_id][float(best_hsp60.threshold)]
    s = score_from_components(
        centroids, areas, gt,
        int(best_hsp60.min_cluster),
        int(best_hsp60.max_cluster),
        MATCH_THRESHOLD,
    )
    s["tomo"] = tomo_id
    best_rows.append(s)

hsp60_best_detail = pd.DataFrame(best_rows)[["tomo","gt","pred","tp","precision","recall","f1"]]
print(hsp60_best_detail.to_string(index=False))

# Save grid
hsp60_grid.to_csv("./deepict/hsp60_pp_grid.csv", index=False)
print("\nSaved hsp60_pp_grid.csv")

hsp60 probability maps found:
  24092001: /virtual_data/train_data/training/deepict/leave_one_out_hsp60_depth2/hsp60d2_loo_0_24092001_out/predictions/hsp60d2_loo_0_24092001_model/24092001/hsp60/probability_map.mrc
  24092002: /virtual_data/train_data/training/deepict/leave_one_out_hsp60_depth2/hsp60d2_loo_1_24092002_out/predictions/hsp60d2_loo_1_24092002_model/24092002/hsp60/probability_map.mrc
  24092018: /virtual_data/train_data/training/deepict/leave_one_out_hsp60_depth2/hsp60d2_loo_2_24092018_out/predictions/hsp60d2_loo_2_24092018_model/24092018/hsp60/probability_map.mrc
  24092024: /virtual_data/train_data/training/deepict/leave_one_out_hsp60_depth2/hsp60d2_loo_3_24092024_out/predictions/hsp60d2_loo_3_24092024_model/24092024/hsp60/probability_map.mrc
  24092028: /virtual_data/train_data/training/deepict/leave_one_out_hsp60_depth2/hsp60d2_loo_4_24092028_out/predictions/hsp60d2_loo_4_24092028_model/24092028/hsp60/probability_map.mrc
  24092031: /virtual_data/train_data/training/deep

In [6]:
# ── Ribosome depth-2 2-fold: probability-map paths ──────────────────────────
from tqdm import tqdm
ribo_base = Path("/virtual_data/train_data/training/deepict/two_fold_ribosome_depth2")

ribo_prob_maps = {}
for fold_out in ribo_base.glob("*_out"):
    for mrc in fold_out.glob("predictions/*/*/ribosome/probability_map.mrc"):
        tomo_id = mrc.parents[1].name
        ribo_prob_maps[tomo_id] = mrc

print("Ribosome probability maps found:")
for k, v in sorted(ribo_prob_maps.items()):
    print(f"  {k}: {v.parent.parent.parent.parent.name} / {k}")

missing_ribo = sorted(set(ribo_centers.keys()) - set(ribo_prob_maps.keys()))
if missing_ribo:
    print(f"\nWARNING - no probability map for: {missing_ribo}")

# ── Ribosome particles are larger (~20 vox) so wider size ranges ─────────────
ribo_thresholds     = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
ribo_min_cluster    = [10, 50, 100, 200]
ribo_max_cluster    = [500, 1000, 3000, 10000]

print("\nBuilding per-threshold component cache (post-process only)...")
ribo_cache = {}
for tomo_id, prob_file in sorted(ribo_prob_maps.items()):
    if tomo_id not in ribo_centers:
        continue
    ribo_cache[tomo_id] = build_threshold_cache(prob_file, ribo_thresholds, connectivity=3)

ribo_grid_records = []
total_r = len(ribo_thresholds) * len(ribo_min_cluster) * len(ribo_max_cluster)
print(f"\nRunning {total_r} parameter combinations for ribosome ...")

for thr, mn, mx in tqdm(product(ribo_thresholds, ribo_min_cluster, ribo_max_cluster)):
    if mn >= mx:
        continue
    micro_tp = micro_fp = micro_fn = 0
    for tomo_id in sorted(ribo_cache.keys()):
        gt = np.array(ribo_centers[tomo_id], dtype=np.float32)
        centroids, areas = ribo_cache[tomo_id][thr]
        s = score_from_components(centroids, areas, gt, mn, mx, MATCH_THRESHOLD)
        micro_tp += s["tp"]
        micro_fp += s["fp"]
        micro_fn += s["fn"]

    p = micro_tp / (micro_tp + micro_fp) if (micro_tp + micro_fp) else 0.0
    r = micro_tp / (micro_tp + micro_fn) if (micro_tp + micro_fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    ribo_grid_records.append(dict(threshold=thr, min_cluster=mn, max_cluster=mx,
                                   micro_tp=micro_tp, micro_fp=micro_fp, micro_fn=micro_fn,
                                   precision=p, recall=r, f1=f1))

ribo_grid = pd.DataFrame(ribo_grid_records).sort_values("f1", ascending=False).reset_index(drop=True)

print("\nTop-20 ribosome parameter combinations by micro-F1:")
print(ribo_grid.head(20)[["threshold","min_cluster","max_cluster","precision","recall","f1"]].to_string(index=False))

best_ribo = ribo_grid.iloc[0]
print(f"\nBest ribosome -> threshold={best_ribo.threshold} min_cluster={int(best_ribo.min_cluster)} "
      f"max_cluster={int(best_ribo.max_cluster)}")
print(f"  micro-F1={best_ribo.f1:.4f} precision={best_ribo.precision:.4f} recall={best_ribo.recall:.4f}")

# ── Per-tomo breakdown with best params ─────────────────────────────────────
print("\nPer-tomo breakdown at best params:")
ribo_best_rows = []
for tomo_id in sorted(ribo_cache.keys()):
    gt = np.array(ribo_centers[tomo_id], dtype=np.float32)
    centroids, areas = ribo_cache[tomo_id][float(best_ribo.threshold)]
    s = score_from_components(
        centroids, areas, gt,
        int(best_ribo.min_cluster),
        int(best_ribo.max_cluster),
        MATCH_THRESHOLD,
    )
    s["tomo"] = tomo_id
    ribo_best_rows.append(s)

ribo_best_detail = pd.DataFrame(ribo_best_rows)[["tomo","gt","pred","tp","precision","recall","f1"]]
print(ribo_best_detail.to_string(index=False))

ribo_grid.to_csv("./deepict/ribo_pp_grid.csv", index=False)
print("\nSaved ribo_pp_grid.csv")

Ribosome probability maps found:
  24092002: predictions / 24092002
  24092009: predictions / 24092009
  24092013: predictions / 24092013
  24092014: predictions / 24092014
  24092016: predictions / 24092016
  24092017: predictions / 24092017
  24092020: predictions / 24092020
  24092023: predictions / 24092023
  24092024: predictions / 24092024
  24092025: predictions / 24092025
  24092026: predictions / 24092026
  24092029: predictions / 24092029
  24092030: predictions / 24092030
  24092031: predictions / 24092031

Building per-threshold component cache (post-process only)...

Running 96 parameter combinations for ribosome ...


96it [00:54,  1.77it/s]



Top-20 ribosome parameter combinations by micro-F1:
 threshold  min_cluster  max_cluster  precision   recall       f1
       0.6          200        10000   0.314491 0.296245 0.305095
       0.6          100        10000   0.302367 0.304590 0.303474
       0.6           50        10000   0.289372 0.311047 0.299818
       0.6           10        10000   0.268241 0.325055 0.293927
       0.5          200        10000   0.232211 0.204252 0.217336
       0.5          100        10000   0.214153 0.205643 0.209811
       0.5           50        10000   0.205361 0.213888 0.209538
       0.5           10        10000   0.183608 0.223227 0.201489
       0.4          200        10000   0.135945 0.117226 0.125894
       0.4          100        10000   0.120966 0.116928 0.118913
       0.4           50        10000   0.111938 0.121101 0.116339
       0.4           10        10000   0.097975 0.131234 0.112192
       0.6           10         3000   0.111720 0.072919 0.088242
       0.6           50

In [7]:
# ── Summary comparison vs. DeepETPicker baseline ────────────────────────────
deepetpicker_hsp60_tomo = {
    "24092001": dict(gt=265, f1=0.5837),
    "24092002": dict(gt=152, f1=0.4537),
    "24092018": dict(gt=136, f1=0.3752),
    "24092024": dict(gt=215, f1=0.4810),
    "24092028": dict(gt=46,  f1=0.2911),
    "24092031": dict(gt=187, f1=0.4567),
}
# micro-F1 approximation: weight by gt count
total_gt_dep = sum(v["gt"] for v in deepetpicker_hsp60_tomo.values())
dep_micro_f1 = sum(v["f1"] * v["gt"] for v in deepetpicker_hsp60_tomo.values()) / total_gt_dep

deepetpicker_ribo_micro_f1 = 0.5260

print("=" * 65)
print("FINAL COMPARISON — micro-F1 summary")
print("=" * 65)
rows = [
    ("DeepETPicker  hsp60",       dep_micro_f1),
    ("DeePiCt depth-2 hsp60 baseline (thr=0.5)", 0.1165),
    ("DeePiCt depth-2 hsp60 BEST (tuned)",        best_hsp60.f1),
    ("DeepETPicker  ribosome",    deepetpicker_ribo_micro_f1),
    ("DeePiCt depth-2 ribo  BEST (tuned)",         best_ribo.f1),
]
for name, f1 in rows:
    print(f"  {name:<48s}  {f1:.4f}")

print()
print("Best hsp60 post-processing params:")
print(f"  threshold     = {best_hsp60.threshold}")
print(f"  min_cluster   = {int(best_hsp60.min_cluster)}")
print(f"  max_cluster   = {int(best_hsp60.max_cluster)}")
print()
print("Best ribosome post-processing params:")
print(f"  threshold     = {best_ribo.threshold}")
print(f"  min_cluster   = {int(best_ribo.min_cluster)}")
print(f"  max_cluster   = {int(best_ribo.max_cluster)}")

FINAL COMPARISON — micro-F1 summary
  DeepETPicker  hsp60                               0.4764
  DeePiCt depth-2 hsp60 baseline (thr=0.5)          0.1165
  DeePiCt depth-2 hsp60 BEST (tuned)                0.2745
  DeepETPicker  ribosome                            0.5260
  DeePiCt depth-2 ribo  BEST (tuned)                0.3051

Best hsp60 post-processing params:
  threshold     = 0.7
  min_cluster   = 20
  max_cluster   = 2000

Best ribosome post-processing params:
  threshold     = 0.6
  min_cluster   = 200
  max_cluster   = 10000


In [3]:
# Progress check after interrupted grid search
done = len(grid_records) if "grid_records" in globals() else 0
total = total if "total" in globals() else 0
pct = (100.0 * done / total) if total else 0.0
print(f"hsp60 grid progress: {done}/{total} ({pct:.2f}%)")

hsp60 grid progress: 1/175 (0.57%)
